# EDA

Exploratory analysis of the KKBox source tables, churn labels and member profiles using pandas.

## Objectives

- Inspect the available files and their structure.
- Measure churn class balance and validate label keys.
- Compare the supplied label versions.
- Assess profile completeness and recorded-age quality.
- Describe transaction dates, payment plans, renewal flags and data quality.
- Inspect daily listening activity and distinguish full-file results from prefix checks.
- Compare pre-cutoff listening and renewal behaviour between supplied outcome groups.

## Data sources

| File | Contents |
|---|---|
| `train.csv` | Customer identifiers and supplied churn labels |
| `train_v2.csv` | A second release of supplied churn labels |
| `members_v3.csv` | Customer profile attributes |
| `transactions.csv`, `transactions_v2.csv` | Subscription payments, renewals and cancellations |
| `user_logs.csv`, `user_logs_v2.csv` | Daily listening activity |

`msno` identifies a customer. The target `is_churn` is an observed outcome supplied by the dataset: `1` indicates churn and `0` indicates non-churn.

**Scope:** previews of all seven sources; full-file label and profile checks; full scans of both transaction files and `user_logs_v2.csv`; and a 100,000-row prefix check of `user_logs.csv`. Versions remain separate. A full streaming pass of the original listening source also retains all pre-cutoff records for a fixed 20,000-customer sample. Recent 7/28/90-day views and personal-baseline comparisons use a working 1 March cutoff; modelling, final eligibility and temporal evaluation remain subsequent work.

## 1. Environment

The selected Jupyter kernel determines where Python executes. The next cell identifies the runtime and configures input and output paths.

| Runtime | Input | Generated output |
|---|---|---|
| Kaggle | `/kaggle/input` (read-only) | `/kaggle/working/churn-eda` |
| Local | `data/raw` | `data/processed/eda` |

When using Kaggle from VS Code, confirm that the cell reports `Python is running on: Kaggle`.

In [ ]:
import platform
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir()
if ON_KAGGLE:
    INPUT = Path("/kaggle/input")
    OUTPUT = Path("/kaggle/working/churn-eda")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    INPUT = ROOT / "data" / "raw"
    OUTPUT = ROOT / "data" / "processed" / "eda"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Python is running on:", "Kaggle" if ON_KAGGLE else "this computer")
print("Python version:", platform.python_version())
print("Data folder:", INPUT)
print("Generated output folder:", OUTPUT)

## 2. Dependencies

`py7zr` extracts the files used for full-file analysis. `libarchive-c` streams small table previews from archives without extracting the full histories. The setup cell installs missing readers; installation requires Internet access. For local use, the project's `notebooks` dependency group includes it:

```bash
python -m pip install -e ".[notebooks]"
```

Files required for full analysis are extracted when a CSV is unavailable. The original listening archive remains compressed; its previews and prefix check stay in memory. `libarchive-c` uses the system libarchive library, available on this Mac and standard Kaggle runtimes.

In [ ]:
import importlib.util
import subprocess
import sys

archive_readers = {"py7zr": "py7zr>=0.20,<2", "libarchive": "libarchive-c>=5,<6"}
for module, requirement in archive_readers.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", requirement])
    else:
        print(f"{module} is already installed.")

## 3. Data access

The helper resolves CSV files from the input directory. If a CSV is available only inside a `.7z` archive, it extracts the matching file to a generated cache and validates its size. Nested archive paths are supported.

On Kaggle, attach the **KKBox churn prediction competition** through the Input panel. The following cell lists available source files.

In [ ]:
import json
from pathlib import PurePosixPath

import matplotlib.pyplot as plt
import pandas as pd
import py7zr
from IPython.display import display


def data_file(filename):
    """Find one source CSV or unpack only its matching archive member."""
    csv_matches = [p for p in INPUT.rglob(filename) if p.is_file()]
    if len(csv_matches) == 1:
        return csv_matches[0]
    if len(csv_matches) > 1:
        raise ValueError(f"Multiple copies of {filename}; choose a single input source.")
    archives = [p for p in INPUT.rglob(filename + ".7z") if p.is_file()]
    if len(archives) != 1:
        raise FileNotFoundError(f"Attach/download the KKBox input containing {filename} or its .7z archive.")
    cache = OUTPUT / "cache" / filename.replace(".csv", "")
    with py7zr.SevenZipFile(archives[0], mode="r") as archive:
        matches = [m for m in archive.getnames() if PurePosixPath(m).name == filename]
        if len(matches) != 1:
            raise ValueError(f"Expected exactly one {filename} in the archive.")
        member = PurePosixPath(matches[0])
        if member.is_absolute() or ".." in member.parts or "\\" in str(member):
            raise ValueError("Unexpected path in archive.")
        destination = cache.joinpath(*member.parts)
        info = next(item for item in archive.list() if item.filename == str(member))
        if not destination.exists():
            cache.mkdir(parents=True, exist_ok=True)
            archive.extract(path=cache, targets=[str(member)])
        if destination.stat().st_size != info.uncompressed:
            raise ValueError(f"Incomplete generated cache for {filename}; remove its cache and rerun.")
    return destination


available = sorted(p.name for p in INPUT.rglob("*") if p.is_file() and p.name.endswith((".csv", ".7z")))
display(pd.DataFrame({"available_source_file": available}))

## 3.1. Source table previews

Display the first five rows of each source table together, before analysing them. Customer IDs are abbreviated in display copies; source values remain unchanged. These prefixes illustrate columns and example values, not population distributions.

CSV previews use `nrows=5`. For compressed inputs, stream only the opening CSV records into memory and stop reading early. This avoids extracting the large listening-history file. Dates remain in their original `YYYYMMDD` form here; versioned histories are shown separately and are not combined.

In [ ]:
import io

import libarchive


def table_head(filename, nrows=5):
    """Read a bounded prefix of one KKBox CSV, including archived sources."""
    csv_paths = [p for p in INPUT.rglob(filename) if p.is_file()]
    if len(csv_paths) == 1:
        return pd.read_csv(csv_paths[0], nrows=nrows, dtype={"msno": "string"})
    if len(csv_paths) > 1:
        raise ValueError(f"Multiple copies of {filename}; choose a single input source.")
    archives = [p for p in INPUT.rglob(filename + ".7z") if p.is_file()]
    if len(archives) != 1:
        raise FileNotFoundError(f"Expected one {filename} or its .7z archive.")
    if nrows < 1 or nrows > 100_000:
        raise ValueError("Archive previews must request between 1 and 100,000 rows.")
    with libarchive.file_reader(str(archives[0])) as archive:
        for entry in archive:
            if PurePosixPath(entry.pathname).name != filename:
                continue
            prefix = bytearray()
            newlines = 0
            for block in entry.get_blocks():
                prefix.extend(block)
                newlines += block.count(b"\n")
                # KKBox source records have one CSV record per line.
                if newlines >= nrows + 1:
                    return pd.read_csv(io.BytesIO(prefix), nrows=nrows,
                                       dtype={"msno": "string"})
                if len(prefix) > 64 * 1024 * 1024:
                    raise ValueError(f"Unexpectedly long records in {filename}.")
            return pd.read_csv(io.BytesIO(prefix), nrows=nrows, dtype={"msno": "string"})
    raise FileNotFoundError(f"No {filename} found inside its archive.")


source_tables = [
    "train.csv", "train_v2.csv", "members_v3.csv",
    "transactions.csv", "transactions_v2.csv", "user_logs.csv", "user_logs_v2.csv",
]
for filename in source_tables:
    preview = table_head(filename).copy()
    preview["msno"] = preview["msno"].str[:6] + "…"
    print(f"{filename} — first 5 rows (IDs abbreviated)")
    display(preview)


## 4. Label data

Load the full `train.csv` into a pandas DataFrame. Customer identifiers remain strings, and `is_churn` uses a nullable integer type. Inspect the dimensions and data types; the source preview is shown in section 3.1.

In [ ]:
train_labels = pd.read_csv(
    data_file("train.csv"),
    dtype={"msno": "string", "is_churn": "Int8"},
)
print("Rows and columns:", train_labels.shape)
print("Column names:", train_labels.columns.tolist())
display(train_labels.dtypes.rename("data_type"))

## 5. Class balance

Count the supplied churn flags and calculate their percentages with `value_counts`.

**Churn rate:** the number of rows labelled `1`, divided by the number of labelled rows, multiplied by 100.

These proportions describe the source population and inform subsequent evaluation design; they are not model performance metrics.

In [ ]:
label_counts = train_labels["is_churn"].value_counts(dropna=False).sort_index()
label_percentages = train_labels["is_churn"].value_counts(normalize=True, dropna=False).sort_index() * 100
label_balance = pd.DataFrame({"customers": label_counts, "percentage": label_percentages})
label_balance.index.name = "is_churn"
display(label_balance)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["0: non-churn", "1: churn"], [int(label_counts.get(0, 0)), int(label_counts.get(1, 0))])
ax.set(title="Supplied outcomes in train.csv", ylabel="Customers")
fig.tight_layout()
fig.savefig(OUTPUT / "label_balance.png", dpi=150)
plt.show()

## 6. Label validation

Check for missing customer identifiers, repeated identifiers and invalid or missing churn flags. Checks apply within `train.csv`; cross-file customer overlap requires a separate assessment of outcome periods.

Rows are counted and reported without applying a cleaning policy.

In [ ]:
label_checks = pd.Series({
    "rows": len(train_labels),
    "unique_customer_ids": train_labels["msno"].nunique(),
    "missing_customer_ids": int(train_labels["msno"].isna().sum()),
    "repeated_customer_ids": int(train_labels["msno"].duplicated().sum()),
    "invalid_or_missing_flags": int((~train_labels["is_churn"].isin([0, 1])).sum()),
})
display(label_checks.rename("count"))

## 7. Label version comparison

Load the full `train_v2.csv`, then compare its row count, churn count and churn rate with `train.csv`.

Establish outcome periods and label availability before assigning training and evaluation datasets. A customer can have different valid outcomes in different periods, so a modelling record will require both a customer identifier and a prediction date. A newer release is not automatically a replacement for an earlier outcome period.

In [ ]:
train_v2_labels = pd.read_csv(
    data_file("train_v2.csv"),
    dtype={"msno": "string", "is_churn": "Int8"},
)
comparison = pd.DataFrame([
    {"source": "train.csv", "rows": len(train_labels),
     "churners": int(train_labels["is_churn"].eq(1).sum()),
     "churn_percentage": train_labels["is_churn"].eq(1).mean() * 100},
    {"source": "train_v2.csv", "rows": len(train_v2_labels),
     "churners": int(train_v2_labels["is_churn"].eq(1).sum()),
     "churn_percentage": train_v2_labels["is_churn"].eq(1).mean() * 100},
])
display(comparison)

## 8. Profile schema

`msno` is the customer join key; `bd` records age, while `gender`, `city` and `registered_via` describe profile attributes. The registration date is stored as `YYYYMMDD`. Example values are shown in section 3.1.

Load the first 1,000 rows for data-type and completeness inspection. This prefix is not a random or representative sample. Population-level quality counts are calculated from the full file in the next section.

In [ ]:
members_path = data_file("members_v3.csv")
members_preview = pd.read_csv(members_path, nrows=1_000, dtype={"msno": "string"})
print("Schema preview ONLY:", members_preview.shape)
display(pd.DataFrame({"data_type": members_preview.dtypes.astype(str),
                      "nonmissing_in_preview": members_preview.notna().sum()}))

## 9. Profile data quality

Read the full profile file in batches of 100,000 rows and accumulate quality counts. `usecols` loads only `bd` and `gender`, limiting memory use.

Report recorded age zero and missing gender as review flags. Their treatment must be agreed before feature engineering; age zero does not automatically justify removing a customer.

In [ ]:
profile_rows = 0
zero_age_rows = 0
missing_gender_rows = 0

for chunk in pd.read_csv(members_path, usecols=["bd", "gender"], chunksize=100_000):
    profile_rows += len(chunk)
    zero_age_rows += int(chunk["bd"].eq(0).sum())
    missing_gender_rows += int(chunk["gender"].isna().sum())

profile_quality = pd.DataFrame([
    {"review_flag": "Recorded age is zero", "customers": zero_age_rows,
     "percentage_of_all_profiles": 100 * zero_age_rows / profile_rows},
    {"review_flag": "Gender is missing", "customers": missing_gender_rows,
     "percentage_of_all_profiles": 100 * missing_gender_rows / profile_rows},
])
print("Full profile file rows:", profile_rows)
display(profile_quality)

## 10. Transaction coverage and quality

A transaction row records a subscription event, not a unique customer. Read each version separately in batches of 100,000 rows; retain aggregate counts rather than the complete history in memory. The first archive extraction can take several minutes.

Parse transaction and expiry dates explicitly as `YYYYMMDD`. Report the date range, missing values, invalid flags, non-positive plan lengths and negative payments. These are review flags; this section does not drop or correct records. Repeated customer IDs are expected because a customer can have many events.

The scan covers all customers and all dates in each source. These histories are descriptive EDA, not a feature table restricted to a prediction cutoff.

In [ ]:
CHUNK_SIZE = 100_000
TRANSACTION_COLUMNS = [
    "msno", "payment_method_id", "payment_plan_days", "plan_list_price",
    "actual_amount_paid", "is_auto_renew", "transaction_date",
    "membership_expire_date", "is_cancel",
]


def scan_transactions(path, source, chunksize=CHUNK_SIZE):
    """Return full-file counts, daily row counts and plan-length frequencies."""
    counts = dict.fromkeys([
        "rows", "rows_with_missing_values", "missing_customer_ids",
        "invalid_transaction_dates", "invalid_expiry_dates",
        "invalid_auto_renew_flags", "invalid_cancel_flags",
        "nonpositive_plan_days", "negative_payment_rows", "zero_payment_rows",
        "auto_renew_rows", "cancel_rows",
    ], 0)
    ranges = {"transaction": [None, None], "expiry": [None, None]}
    daily_rows = pd.Series(dtype="int64")
    plan_counts = pd.Series(dtype="int64")
    dtypes = {"msno": "string", "transaction_date": "string",
              "membership_expire_date": "string"}
    for chunk in pd.read_csv(path, usecols=TRANSACTION_COLUMNS,
                             dtype=dtypes, chunksize=chunksize):
        counts["rows"] += len(chunk)
        counts["rows_with_missing_values"] += int(chunk.isna().any(axis=1).sum())
        counts["missing_customer_ids"] += int(chunk["msno"].isna().sum())
        dates = pd.to_datetime(chunk["transaction_date"], format="%Y%m%d", errors="coerce")
        expiries = pd.to_datetime(chunk["membership_expire_date"],
                                  format="%Y%m%d", errors="coerce")
        counts["invalid_transaction_dates"] += int(dates.isna().sum())
        counts["invalid_expiry_dates"] += int(expiries.isna().sum())
        for name, values in [("transaction", dates), ("expiry", expiries)]:
            if values.notna().any():
                low, high = values.min(), values.max()
                ranges[name][0] = low if ranges[name][0] is None else min(ranges[name][0], low)
                ranges[name][1] = high if ranges[name][1] is None else max(ranges[name][1], high)
        counts["invalid_auto_renew_flags"] += int((~chunk["is_auto_renew"].isin([0, 1])).sum())
        counts["invalid_cancel_flags"] += int((~chunk["is_cancel"].isin([0, 1])).sum())
        counts["nonpositive_plan_days"] += int(chunk["payment_plan_days"].le(0).sum())
        counts["negative_payment_rows"] += int(chunk["actual_amount_paid"].lt(0).sum())
        counts["zero_payment_rows"] += int(chunk["actual_amount_paid"].eq(0).sum())
        counts["auto_renew_rows"] += int(chunk["is_auto_renew"].eq(1).sum())
        counts["cancel_rows"] += int(chunk["is_cancel"].eq(1).sum())
        daily_rows = daily_rows.add(dates.value_counts(), fill_value=0)
        plan_counts = plan_counts.add(chunk["payment_plan_days"].value_counts(), fill_value=0)
        if counts["rows"] % 5_000_000 == 0:
            print(f"{source}: scanned {counts['rows']:,} rows")
    result = {"source": source, "scope": "Full file", **counts}
    for name, (low, high) in ranges.items():
        result[f"first_{name}_date"] = None if low is None else low.date().isoformat()
        result[f"last_{name}_date"] = None if high is None else high.date().isoformat()
    return result, daily_rows.sort_index().astype("int64"), plan_counts.astype("int64")


transaction_results = []
transaction_daily = {}
transaction_plans = {}
for filename in ["transactions.csv", "transactions_v2.csv"]:
    print("Scanning:", filename)
    result, daily, plans = scan_transactions(data_file(filename), filename)
    transaction_results.append(result)
    transaction_daily[filename] = daily
    transaction_plans[filename] = plans
transaction_summary = pd.DataFrame(transaction_results)
display(transaction_summary[[
    "source", "scope", "rows", "first_transaction_date", "last_transaction_date",
    "first_expiry_date", "last_expiry_date",
]])
quality_columns = [
    "rows_with_missing_values", "missing_customer_ids", "invalid_transaction_dates",
    "invalid_expiry_dates", "invalid_auto_renew_flags", "invalid_cancel_flags",
    "nonpositive_plan_days", "negative_payment_rows",
]
display(transaction_summary.set_index("source")[quality_columns].T)


## 11. Payment and renewal patterns

Compare the share of transaction rows marked auto-renew, cancellation or zero payment, then show the ten most frequent plan lengths in each file. Percentages use all transaction rows in that file as their denominator; they do not describe the percentage of customers. The flags can overlap.

`is_cancel=1` records a cancellation event. It is not the supplied churn label: a customer can cancel a plan and subsequently renew. Zero payments and non-positive plan lengths require interpretation rather than automatic deletion.

Plot the daily number of transaction rows separately for each version. A date range or chart does not establish whether overlapping versions contain duplicate or revised events; reconciliation is required before combining them.

In [ ]:
payment_patterns = transaction_summary[["source", "rows"]].copy()
for count_column in ["auto_renew_rows", "cancel_rows", "zero_payment_rows"]:
    payment_patterns[count_column.replace("_rows", "_percentage")] = (
        100 * transaction_summary[count_column] / transaction_summary["rows"]
    )
display(payment_patterns)

for filename, counts in transaction_plans.items():
    top_plans = counts.sort_values(ascending=False).head(10).rename_axis("plan_days")
    print(filename, "— ten most frequent plan lengths")
    display(top_plans.rename("transaction_rows").to_frame())

fig, axes = plt.subplots(2, 1, figsize=(10, 6))
for ax, (filename, daily) in zip(axes, transaction_daily.items()):
    ax.plot(daily.index, daily.values, linewidth=1)
    ax.set(title=filename, ylabel="Transaction rows", xlabel="Transaction date")
fig.tight_layout()
fig.savefig(OUTPUT / "transaction_activity.png", dpi=150)
plt.show()


## 12. Listening coverage and quality

A listening row is intended to describe one customer on one day. Check dates, missing values and negative counts or durations before interpreting activity. Full-file customer/date uniqueness is not tested here.

| Source | Scan | Interpretation |
|---|---|---|
| `user_logs_v2.csv` | Full file, read in batches | Coverage and totals describe this complete source |
| `user_logs.csv` | First 100,000 rows, streamed from the archive if necessary | Structural and quality check only; not a random sample |

The original prefix cannot establish historical date coverage, population averages or a trend relative to the updated file. It is deliberately excluded from the activity charts. Negative durations or play counts are reported without being corrected. Raw totals include flagged values; the quality table must be read alongside them.

In [ ]:
LISTENING_COLUMNS = [
    "msno", "date", "num_25", "num_50", "num_75", "num_985", "num_100",
    "num_unq", "total_secs",
]
PLAY_COLUMNS = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq"]
ORIGINAL_LOG_PREFIX_ROWS = 100_000


def scan_listening(chunks, source, scope):
    """Summarise supplied chunks without retaining customer-level history."""
    counts = dict.fromkeys([
        "rows", "rows_with_missing_values", "missing_customer_ids", "invalid_dates",
        "negative_duration_rows", "negative_play_count_rows", "zero_duration_rows",
    ], 0)
    first_date = last_date = None
    daily_parts = []
    for chunk in chunks:
        counts["rows"] += len(chunk)
        counts["rows_with_missing_values"] += int(chunk.isna().any(axis=1).sum())
        counts["missing_customer_ids"] += int(chunk["msno"].isna().sum())
        dates = pd.to_datetime(chunk["date"].astype("string"),
                               format="%Y%m%d", errors="coerce")
        counts["invalid_dates"] += int(dates.isna().sum())
        counts["negative_duration_rows"] += int(chunk["total_secs"].lt(0).sum())
        counts["negative_play_count_rows"] += int(chunk[PLAY_COLUMNS].lt(0).any(axis=1).sum())
        counts["zero_duration_rows"] += int(chunk["total_secs"].eq(0).sum())
        if dates.notna().any():
            low, high = dates.min(), dates.max()
            first_date = low if first_date is None else min(first_date, low)
            last_date = high if last_date is None else max(last_date, high)
        daily = chunk.assign(parsed_date=dates).groupby("parsed_date").agg(
            log_rows=("total_secs", "size"),
            duration_values=("total_secs", "count"),
            total_secs=("total_secs", "sum"),
        )
        daily_parts.append(daily)
        if counts["rows"] % 5_000_000 == 0:
            print(f"{source}: scanned {counts['rows']:,} rows")
    daily_totals = (pd.concat(daily_parts).groupby(level=0).sum().sort_index()
                    if daily_parts else pd.DataFrame(
                        columns=["log_rows", "duration_values", "total_secs"]))
    result = {
        "source": source, "scope": scope, **counts,
        "first_observed_date": None if first_date is None else first_date.date().isoformat(),
        "last_observed_date": None if last_date is None else last_date.date().isoformat(),
    }
    return result, daily_totals


updated_log_chunks = pd.read_csv(
    data_file("user_logs_v2.csv"), usecols=LISTENING_COLUMNS,
    dtype={"msno": "string", "date": "string"}, chunksize=CHUNK_SIZE,
)
updated_log_result, listening_daily = scan_listening(
    updated_log_chunks, "user_logs_v2.csv", "Full file",
)
original_log_prefix = table_head("user_logs.csv", nrows=ORIGINAL_LOG_PREFIX_ROWS)
original_log_result, _ = scan_listening(
    [original_log_prefix], "user_logs.csv", f"First {ORIGINAL_LOG_PREFIX_ROWS:,} rows only",
)
del original_log_prefix
listening_summary = pd.DataFrame([updated_log_result, original_log_result])
display(listening_summary[[
    "source", "scope", "rows", "first_observed_date", "last_observed_date",
]])
display(listening_summary.set_index(["source", "scope"])[[
    "rows_with_missing_values", "missing_customer_ids", "invalid_dates",
    "negative_duration_rows", "negative_play_count_rows", "zero_duration_rows",
]].T)


## 13. Daily listening activity

Use only the full updated listening file for the following table and charts. The first chart counts log rows per day; this is not a verified count of unique active customers. The second shows average recorded minutes per non-missing duration value on each day.

These are source-wide descriptive patterns. They do not compare churn groups or establish predictive or causal relationships. Missing or invalid dates are excluded from daily groups and counted in section 12.

In [ ]:
listening_daily["mean_recorded_minutes"] = (
    listening_daily["total_secs"] / listening_daily["duration_values"].replace(0, float("nan")) / 60
)
print("user_logs_v2.csv — first five daily aggregates")
display(listening_daily.head())

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(listening_daily.index, listening_daily["log_rows"], marker=".")
axes[0].set(title="Daily listening records — user_logs_v2.csv", ylabel="Log rows")
axes[1].plot(listening_daily.index, listening_daily["mean_recorded_minutes"], marker=".")
axes[1].set(ylabel="Mean recorded minutes per row", xlabel="Listening date")
fig.tight_layout()
fig.savefig(OUTPUT / "listening_activity.png", dpi=150)
plt.show()


## 14. Question and study design

**Question:** how do recent activity, longer-term habits and renewal events differ between customers with different later supplied outcomes?

The prediction cutoff and the history windows are separate choices. The cutoff defines what was knowable; the windows define which parts of that past we summarise. Retain all pre-cutoff history, then calculate several views of it.

| Choice | Working setting and reason |
|---|---|
| Prediction cutoff | Start of **1 March 2017**; exclude events on or after this date |
| Listening and transaction history | All available earlier records in the original source files |
| Recent views | **7, 28 and 90 days**: recent activity, four-week habits and broader context |
| Personal baseline | Latest **28 days** versus the preceding **62 days**, using recorded minutes per day |
| Observation population | Same fixed random sample of up to **20,000** IDs from `train_v2.csv`; seed **42** |
| Outcome use | Join supplied labels only after the history summaries are built |

Keeping the same IDs and cutoff lets us examine what different windows add. The sample makes exploration manageable; it does not fix the eventual training population. The windows are interpretable candidates, not proven optimal settings. Any later choice based on predictive performance belongs inside training/validation, with a separate final test.

Longer history provides context, but a larger lifetime total can simply reflect a longer observation period. Report observed history length and coverage. A first recorded listening event is not a verified signup date, and absent records do not prove inactivity.

This remains retrospective EDA of the supplied cohort. Event dates do not prove ingestion availability, cohort eligibility at the cutoff is unverified, and label maturity must be resolved before temporal model evaluation. Updated releases remain separate until their overlap and availability are reconciled.

In [ ]:
PREDICTION_DATE = pd.Timestamp("2017-03-01")
RECENT_START = PREDICTION_DATE - pd.Timedelta(days=28)
BASELINE_START = PREDICTION_DATE - pd.Timedelta(days=90)
LOOKBACK_DAYS = (7, 28, 90)
COHORT_SAMPLE_SIZE = 20_000
COHORT_SEED = 42

assert train_v2_labels["msno"].notna().all()
assert train_v2_labels["msno"].is_unique
assert train_v2_labels["is_churn"].isin([0, 1]).all()
cohort_ids = train_v2_labels[["msno"]].sample(
    n=min(COHORT_SAMPLE_SIZE, len(train_v2_labels)), random_state=COHORT_SEED,
).sort_values("msno").reset_index(drop=True)
sampled_ids = set(cohort_ids["msno"])
cohort_labels = cohort_ids.merge(train_v2_labels, on="msno", validate="one_to_one")
print("Comparison customers:", len(cohort_ids))
print("Prediction cutoff:", PREDICTION_DATE.date())
print("Retained history: all earlier original-source records")
print("Recent windows:", LOOKBACK_DAYS, "days")


## 15. Retain all earlier listening history

Stream the entire original listening source in bounded-memory batches and retain all rows before the cutoff for the fixed customer sample. The archive remains compressed on disk. Unlike the earlier February-only selection, this cache supports new lookback windows without another full-source scan.

The first all-history pass can take around 15–20 minutes on this Mac. Matching reruns reuse the generated cache. The manifest records the source size and modification time, cutoff, seed and hash of the sampled IDs. It identifies this cache as **all pre-cutoff history**, so the earlier February-only cache cannot be mistaken for it.

The helper cell handles file access. The following cell performs the selection and checks its boundaries. Cached customer records contain no labels and remain outside Git.

In [ ]:
import hashlib


def find_source(input_dir, filename):
    """Resolve one CSV, otherwise one archive; never extract history to disk."""
    for name in [filename, filename + ".7z"]:
        matches = [p for p in input_dir.rglob(name) if p.is_file()]
        if len(matches) > 1:
            raise ValueError(f"Multiple copies of {name}; choose one source.")
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Missing {filename} or its archive.")


class ArchiveCSVStream(io.RawIOBase):
    """Expose decompressed blocks to pandas using bounded memory."""

    def __init__(self, blocks):
        super().__init__()
        self.blocks = iter(blocks)
        self.pending = memoryview(b"")

    def readable(self):
        return True

    def readinto(self, target):
        while not self.pending:
            try:
                self.pending = memoryview(next(self.blocks))
            except StopIteration:
                return 0
        count = min(len(target), len(self.pending))
        target[:count] = self.pending[:count]
        self.pending = self.pending[count:]
        return count


def source_chunks(input_dir, filename, usecols, dtype, chunksize=500_000):
    """Yield pandas chunks from a CSV or a streamed archive member."""
    path = find_source(input_dir, filename)
    if path.suffix != ".7z":
        with pd.read_csv(path, usecols=usecols, dtype=dtype, chunksize=chunksize) as reader:
            yield from reader
        return
    with libarchive.file_reader(str(path)) as archive:
        for entry in archive:
            if PurePosixPath(entry.pathname).name == filename:
                with (
                    io.BufferedReader(ArchiveCSVStream(entry.get_blocks(1024 * 1024))) as stream,
                    pd.read_csv(stream, usecols=usecols, dtype=dtype, chunksize=chunksize) as reader,
                ):
                    yield from reader
                return
    raise FileNotFoundError(f"No {filename} inside {path.name}.")


def sampled_listening_all(input_dir, output_dir, customer_ids, cutoff, seed):
    """Cache only sampled pre-cutoff rows; labels are never inputs to filtering."""
    source = find_source(input_dir, "user_logs.csv")
    cache_dir = output_dir / "cohort_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / "precutoff_listening_sample.csv"
    manifest_path = cache_dir / "precutoff_listening_manifest.json"
    ordered_ids = sorted(customer_ids)
    fingerprint = {
        "version": 2, "history_scope": "all_pre_cutoff", "source": str(source.resolve()), "source_size": source.stat().st_size,
        "source_mtime_ns": source.stat().st_mtime_ns, "cutoff": cutoff.isoformat(), "seed": seed, "sample_size": len(ordered_ids),
        "sample_ids_sha256": hashlib.sha256("\n".join(ordered_ids).encode()).hexdigest(),
    }
    dtypes = {"msno": "string", "date": "string", "total_secs": "float64"}
    if manifest_path.exists() and cache_path.exists():
        saved = json.loads(manifest_path.read_text())
        if saved.get("fingerprint") == fingerprint:
            cached = pd.read_csv(cache_path, dtype=dtypes)
            if len(cached) == saved["selected_rows"]:
                print("Using cached all-history listening sample; full source scan already completed.")
                return cached, saved
    selected = []
    scanned = 0
    end_key = cutoff.strftime("%Y%m%d")
    for chunk in source_chunks(input_dir, "user_logs.csv", list(dtypes), dtypes):
        scanned += len(chunk)
        keep = chunk["date"].lt(end_key) & chunk["msno"].isin(customer_ids)
        if keep.any():
            selected.append(chunk.loc[keep].copy())
        if scanned % 10_000_000 == 0:
            print(f"Original listening history: scanned {scanned:,} rows", flush=True)
    history = (pd.concat(selected, ignore_index=True) if selected
               else pd.DataFrame({k: pd.Series(dtype=v) for k, v in dtypes.items()}))
    manifest = {"fingerprint": fingerprint, "scanned_rows": scanned,
                "selected_rows": len(history), "full_source_scan": True}
    temporary = cache_path.with_suffix(".tmp")
    history.to_csv(temporary, index=False)
    temporary.replace(cache_path)
    temporary = manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(manifest, indent=2))
    temporary.replace(manifest_path)
    print(f"Full scan complete: {scanned:,} source rows; kept {len(history):,} sampled rows.")
    return history, manifest


In [ ]:
cohort_listening, cohort_scan_manifest = sampled_listening_all(
    INPUT, OUTPUT, sampled_ids, PREDICTION_DATE, COHORT_SEED,
)
cohort_listening["parsed_date"] = pd.to_datetime(
    cohort_listening["date"], format="%Y%m%d", errors="coerce",
)
assert cohort_listening["msno"].isin(sampled_ids).all()
assert cohort_listening["parsed_date"].dropna().lt(PREDICTION_DATE).all()
# Unparseable dates are excluded and counted in the quality summary below.
assert cohort_scan_manifest["fingerprint"]["history_scope"] == "all_pre_cutoff"
display(pd.Series({
    "source_rows_scanned": cohort_scan_manifest["scanned_rows"],
    "sampled_pre_cutoff_rows": len(cohort_listening),
    "sampled_customers_with_any_records": cohort_listening["msno"].nunique(),
    "sampled_customers_total": len(cohort_ids),
    "first_sampled_date": cohort_listening["parsed_date"].min(),
    "last_sampled_date": cohort_listening["parsed_date"].max(),
}).rename("value"))


## 16. Summarise activity, coverage and change

Build one label-free row per sampled customer. Each summary answers a different question:

| Summary | Interpretation |
|---|---|
| Recorded active days in 7 / 28 / 90 days | How often does this source record positive-duration listening recently? |
| All-history totals and observed span | How much history is available, and over what length of time? |
| Days since last positive log | How long since the latest recorded positive-duration listening? |
| History coverage category | Are there recent records, older records only, or no earlier source records? |
| Recent-to-baseline ratio | Has recorded listening intensity changed relative to this customer's earlier history? |

**Personal baseline:** divide recent 28-day minutes by 28 and earlier 62-day minutes by 62. A ratio of **1** means equal daily recorded intensity; **0.5** means half as much; **2** means twice as much. Calculate it only for customers first observed at least 90 days earlier, with no unusable records in that 90-day period and a positive baseline. This observation-span rule is a proxy for coverage, not proof of continuous observation or customer tenure.

Keep the same quality policy: exclude all repeated customer/day rows and negative, missing or infinite durations; report exclusions. Missing duration totals remain missing in the window table. For the defined baseline comparison only, absent window records contribute zero **recorded** minutes; this does not establish zero real listening. A zero baseline gives an undefined ratio, not infinity.

The earlier two-half February change is retained as a reference. These rules describe exploratory summaries; they do not clean or alter raw data.

In [ ]:
def listening_windows(history, customer_ids, cutoff, windows=(7, 28, 90)):
    """Build label-free listening summaries, with coverage and quality reported."""
    dates = pd.to_datetime(history["date"], format="%Y%m%d", errors="coerce")
    selected_ids = history["msno"].isin(customer_ids)
    keep = selected_ids & dates.lt(cutoff)
    rows = history.loc[keep].copy()
    rows["parsed_date"] = dates.loc[keep]
    duplicate_day = rows.duplicated(["msno", "parsed_date"], keep=False)
    valid_duration = (rows["total_secs"].notna() & rows["total_secs"].ge(0)
                      & ~rows["total_secs"].isin([float("inf"), float("-inf")]))
    unusable = duplicate_day | ~valid_duration
    valid = rows.loc[~unusable].copy()
    positive = valid.loc[valid["total_secs"].gt(0)]
    result = pd.DataFrame(index=pd.Index(sorted(customer_ids), name="msno"))

    def counts(frame, column=None):
        grouped = frame.groupby("msno")
        values = grouped.size() if column is None else grouped[column].nunique()
        return values.reindex(result.index, fill_value=0)

    def minutes(frame):
        return frame.groupby("msno")["total_secs"].sum().reindex(result.index) / 60

    result["first_valid_log_date"] = valid.groupby("msno")["parsed_date"].min().reindex(result.index)
    result["observed_history_span_days"] = (cutoff - result["first_valid_log_date"]).dt.days
    last_positive = positive.groupby("msno")["parsed_date"].max().reindex(result.index)
    result["days_since_last_positive_log"] = (cutoff - last_positive).dt.days
    for suffix, days in [("all", None), *[(f"{n}d", n) for n in windows]]:
        start = None if days is None else cutoff - pd.Timedelta(days=days)
        raw_window = rows if start is None else rows.loc[rows["parsed_date"].ge(start)]
        valid_window = valid if start is None else valid.loc[valid["parsed_date"].ge(start)]
        positive_window = positive if start is None else positive.loc[positive["parsed_date"].ge(start)]
        result[f"source_rows_{suffix}"] = counts(raw_window)
        result[f"valid_days_{suffix}"] = counts(valid_window, "parsed_date")
        result[f"recorded_active_days_{suffix}"] = counts(positive_window, "parsed_date")
        result[f"recorded_minutes_{suffix}"] = minutes(valid_window)

    recent_start = cutoff - pd.Timedelta(days=28)
    baseline_start = cutoff - pd.Timedelta(days=90)
    midpoint = cutoff - pd.Timedelta(days=14)
    recent = valid.loc[valid["parsed_date"].ge(recent_start)]
    baseline = valid.loc[valid["parsed_date"].ge(baseline_start)
                         & valid["parsed_date"].lt(recent_start)]
    unusable_90_ids = set(rows.loc[unusable & rows["parsed_date"].ge(baseline_start), "msno"])
    result["observed_before_90d"] = result["first_valid_log_date"].le(baseline_start)
    result["has_unusable_records_90d"] = result.index.isin(unusable_90_ids)
    eligible = result["observed_before_90d"] & ~result["has_unusable_records_90d"]
    result["baseline_comparison_eligible"] = eligible
    # Rates describe recorded source activity. Absent window records contribute
    # zero only to this explicitly defined, older-observed comparison group.
    result["recent_recorded_minutes_per_day"] = (minutes(recent).fillna(0) / 28).where(eligible)
    result["baseline_recorded_minutes_per_day"] = (minutes(baseline).fillna(0) / 62).where(eligible)
    baseline_rate = result["baseline_recorded_minutes_per_day"].replace(0, float("nan"))
    result["recent_to_baseline_ratio"] = result["recent_recorded_minutes_per_day"] / baseline_rate

    early_ids = set(recent.loc[recent["parsed_date"].lt(midpoint), "msno"])
    late_ids = set(recent.loc[recent["parsed_date"].ge(midpoint), "msno"])
    result["observed_both_halves"] = result.index.isin(early_ids & late_ids)
    early_days = counts(positive.loc[positive["parsed_date"].ge(recent_start)
                                       & positive["parsed_date"].lt(midpoint)], "parsed_date")
    late_days = counts(positive.loc[positive["parsed_date"].ge(midpoint)], "parsed_date")
    result["recorded_day_change_when_observed"] = (late_days - early_days).where(result["observed_both_halves"])
    result["history_coverage"] = "No pre-cutoff source records"
    result.loc[result["source_rows_all"].gt(0), "history_coverage"] = "Older records only; none in last 28 days"
    result.loc[result["source_rows_28d"].gt(0), "history_coverage"] = "Source records in last 28 days"
    quality = {
        "cached_rows": len(history), "selected_pre_cutoff_rows": len(rows),
        "invalid_date_rows": int((selected_ids & dates.isna()).sum()),
        "rows_on_or_after_cutoff_excluded": int((selected_ids & dates.ge(cutoff)).sum()),
        "rows_excluded_for_duplicate_day_keys": int(duplicate_day.sum()),
        "rows_with_invalid_duration": int((~valid_duration).sum()),
        "usable_listening_rows": len(valid),
        "customers_without_any_source_records": int(result["source_rows_all"].eq(0).sum()),
        "customers_without_recent_source_records": int(result["source_rows_28d"].eq(0).sum()),
        "customers_older_records_only": int(result["history_coverage"].eq("Older records only; none in last 28 days").sum()),
        "first_selected_date": None if rows.empty else rows["parsed_date"].min().date().isoformat(),
        "last_selected_date": None if rows.empty else rows["parsed_date"].max().date().isoformat(),
    }
    return result, quality


cohort_features, cohort_listening_quality = listening_windows(
    cohort_listening, sampled_ids, PREDICTION_DATE, LOOKBACK_DAYS,
)
display(pd.Series(cohort_listening_quality).rename("value"))
assert "is_churn" not in cohort_features.columns
for days in LOOKBACK_DAYS:
    assert cohort_features[f"recorded_active_days_{days}d"].between(0, days).all()


## 17. Summarise earlier subscription events

Use all original transactions before the cutoff. Count transaction and cancellation events over the same 7 / 28 / 90-day windows and all earlier history, and calculate time since the latest transaction. These are event counts, not unique subscription counts or a reconstruction of membership state.

Inspect the auto-renew flag on the latest recorded transaction date. Conflicting or invalid same-day flags remain ambiguous; customers without earlier events form a separate category. The latest event flag is not a verified current subscription state, and a cancellation event is not the churn label. Labels are still separate from these summaries.

In [ ]:
def renewal_by_customer(chunks, customer_ids, cutoff):
    """Summarise all prior events and recent windows, preserving uncertain flags."""
    selected = []
    for chunk in chunks:
        rows = chunk.loc[chunk["msno"].isin(customer_ids)].copy()
        rows["parsed_date"] = pd.to_datetime(rows["transaction_date"],
                                             format="%Y%m%d", errors="coerce")
        rows = rows.loc[rows["parsed_date"].lt(cutoff)]
        if not rows.empty:
            selected.append(rows)
    result = pd.DataFrame(index=pd.Index(sorted(customer_ids), name="msno"))
    result["latest_auto_renew_status"] = "No earlier transaction"
    for suffix in ["all", "7d", "28d", "90d"]:
        result[f"transaction_events_{suffix}"] = 0
        result[f"cancellation_events_{suffix}"] = 0
    result["days_since_last_transaction"] = float("nan")
    if not selected:
        return result
    events = pd.concat(selected, ignore_index=True)
    last_dates = events.groupby("msno")["parsed_date"].transform("max")
    latest = events.loc[events["parsed_date"].eq(last_dates)]

    def flag_status(values):
        if not values.isin([0, 1]).all() or values.nunique() != 1:
            return "Ambiguous latest-day flags"
        return "Auto-renew 1" if values.iloc[0] == 1 else "Auto-renew 0"

    statuses = latest.groupby("msno")["is_auto_renew"].agg(flag_status)
    result.loc[statuses.index, "latest_auto_renew_status"] = statuses
    last_by_customer = latest.groupby("msno")["parsed_date"].max().reindex(result.index)
    result["days_since_last_transaction"] = (cutoff - last_by_customer).dt.days
    for suffix, days in [("all", None), ("7d", 7), ("28d", 28), ("90d", 90)]:
        frame = events if days is None else events.loc[
            events["parsed_date"].ge(cutoff - pd.Timedelta(days=days))]
        result[f"transaction_events_{suffix}"] = frame.groupby("msno").size().reindex(result.index, fill_value=0)
        cancellations = frame.loc[frame["is_cancel"].eq(1)]
        result[f"cancellation_events_{suffix}"] = cancellations.groupby("msno").size().reindex(result.index, fill_value=0)
    return result


renewal_chunks = pd.read_csv(
    data_file("transactions.csv"),
    usecols=["msno", "transaction_date", "is_auto_renew", "is_cancel"],
    dtype={"msno": "string", "transaction_date": "string"}, chunksize=CHUNK_SIZE,
)
cohort_renewal = renewal_by_customer(renewal_chunks, sampled_ids, PREDICTION_DATE)
cohort_features = cohort_features.join(cohort_renewal, validate="one_to_one")
cohort_features["prediction_date"] = PREDICTION_DATE.date().isoformat()
assert "is_churn" not in cohort_features.columns
print("Label-free customer summaries:", cohort_features.shape)


## 18. Compare outcomes with denominators visible

Join the supplied labels now, with one row per sampled customer.

1. **Window table:** compare churn groups within each window. Different-length windows naturally contain different totals; a larger 90-day count than a 7-day count is not evidence of a behavioural change. Duration medians use customers with valid records, whose count is shown.
2. **Coverage table:** distinguish older records with no recent records from having no earlier records at all. The latter is an unknown-history group, not an established inactive group.
3. **Customer comparison:** inspect recency, observed history span and personal-baseline ratios. The ratio denominator is the number of customers eligible for that calculation, not the whole sample.
4. **Renewal table:** inspect the proportion labelled churn within each latest-event category, alongside category size.

These are exploratory associations in a retrospectively selected cohort. Similar medians do not establish that a variable is useless; different medians do not establish causation or model performance. Keep final model evaluation separate from these exploratory choices.

In [ ]:
cohort_comparison = cohort_features.reset_index().merge(
    cohort_labels, on="msno", validate="one_to_one",
)
assert len(cohort_comparison) == len(cohort_ids)
cohort_comparison["outcome"] = cohort_comparison["is_churn"].map({0: "Non-churn", 1: "Churn"})
cohort_comparison["has_source_records"] = cohort_comparison["source_rows_28d"].gt(0)
cohort_comparison["has_any_source_records"] = cohort_comparison["source_rows_all"].gt(0)
cohort_comparison["has_cancellation_event"] = cohort_comparison["cancellation_events_28d"].gt(0)

window_parts = []
for suffix in ["7d", "28d", "90d", "all"]:
    frame = cohort_comparison.assign(has_window_records=cohort_comparison[f"source_rows_{suffix}"].gt(0))
    part = frame.groupby("outcome").agg(
        customers=("is_churn", "size"), customers_with_source_records=("has_window_records", "sum"),
        customers_with_valid_duration=(f"recorded_minutes_{suffix}", "count"),
        median_recorded_active_days=(f"recorded_active_days_{suffix}", "median"),
        median_recorded_minutes_valid_only=(f"recorded_minutes_{suffix}", "median"),
    ).reset_index()
    part.insert(0, "history_window", suffix)
    window_parts.append(part)
window_comparison = pd.concat(window_parts, ignore_index=True)
print("Recorded activity by history window and supplied outcome")
display(window_comparison)

coverage_comparison = cohort_comparison.groupby(["outcome", "history_coverage"]).size().rename("customers").to_frame()
coverage_comparison["percentage_within_outcome"] = (
    100 * coverage_comparison["customers"] / coverage_comparison["customers"].groupby(level="outcome").transform("sum")
)
print("Recent, older-only and absent source history")
display(coverage_comparison)

outcome_comparison = cohort_comparison.groupby("outcome").agg(
    customers=("is_churn", "size"),
    customers_with_source_records=("has_source_records", "sum"),
    customers_with_any_source_records=("has_any_source_records", "sum"),
    customers_with_valid_duration=("recorded_minutes_28d", "count"),
    customers_observed_both_halves=("observed_both_halves", "sum"),
    median_recorded_active_days_28d=("recorded_active_days_28d", "median"),
    median_recorded_minutes_28d=("recorded_minutes_28d", "median"),
    median_day_change_when_observed=("recorded_day_change_when_observed", "median"),
    median_observed_history_span_days=("observed_history_span_days", "median"),
    median_days_since_last_positive_log=("days_since_last_positive_log", "median"),
    customers_eligible_for_baseline=("baseline_comparison_eligible", "sum"),
    customers_with_defined_ratio=("recent_to_baseline_ratio", "count"),
    median_recent_to_baseline_ratio=("recent_to_baseline_ratio", "median"),
    customers_with_cancellation_event=("has_cancellation_event", "sum"),
)
outcome_comparison["source_record_coverage_percentage"] = (
    100 * outcome_comparison["customers_with_source_records"] / outcome_comparison["customers"]
)
outcome_comparison["cancellation_event_percentage"] = (
    100 * outcome_comparison["customers_with_cancellation_event"] / outcome_comparison["customers"]
)
print("Customer history and personal-baseline comparison")
display(outcome_comparison.T)

renewal_comparison = cohort_comparison.groupby("latest_auto_renew_status").agg(
    customers=("is_churn", "size"), churners=("is_churn", "sum"),
)
renewal_comparison["churn_percentage"] = 100 * renewal_comparison["churners"] / renewal_comparison["customers"]
print("Later supplied outcomes by latest pre-cutoff auto-renew flag")
display(renewal_comparison)


## 19. Visualise recent windows and personal change

The first three panels compare recorded listening days within a fixed window. Each includes all sampled customers, so source-coverage differences remain relevant. Raw counts across different window lengths should not be interpreted as a trend.

The fourth panel compares the personal-baseline ratio for eligible customers with a positive earlier baseline. The dashed line at 1 indicates equal recorded minutes per day in the two periods. Read its sample counts in section 18.

Boxes show the median and middle half of values. Outlier markers are hidden for readability; no records are removed by that display choice. All-history totals remain in the table with observation-span information, since lifetime exposure differs between customers.

In [ ]:
outcome_order = ["Non-churn", "Churn"]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, days in zip(axes.flat, LOOKBACK_DAYS):
    ax.boxplot([
        cohort_comparison.loc[cohort_comparison["outcome"].eq(group), f"recorded_active_days_{days}d"].dropna()
        for group in outcome_order
    ], tick_labels=outcome_order, showfliers=False)
    ax.set(title=f"Last {days} days", ylabel="Recorded positive-duration days")
axes[1, 1].boxplot([
    cohort_comparison.loc[cohort_comparison["outcome"].eq(group), "recent_to_baseline_ratio"].dropna()
    for group in outcome_order
], tick_labels=outcome_order, showfliers=False)
axes[1, 1].axhline(1, color="grey", linestyle="--", linewidth=1)
axes[1, 1].set(title="Recent 28 days / earlier 62 days", ylabel="Ratio of recorded minutes per day")
fig.suptitle("History before 1 March; fixed customer sample; supplied later outcomes")
fig.tight_layout()
fig.savefig(OUTPUT / "cohort_listening_comparison.png", dpi=150)
plt.show()


## 20. Findings and decisions

The next cell assembles findings directly from the calculated counts. These are data observations, not model results. The outcome-group comparisons use the fixed sample and pre-cutoff history described above. Review the source scope beside every finding: the original listening prefix is not representative of the full file.

Before building `02_features.ipynb`, agree:

1. **Population and label period:** confirm whether the working EDA population and supplied outcomes are suitable for the first model.
2. **Prediction moment:** review the working 1 March cutoff, source availability assumptions and when training labels become known.
3. **Source reconciliation:** how to handle overlapping transaction releases and repeated customer/date listening records.
4. **Quality policy:** how to treat age zero, missing profiles, non-positive plan lengths, unusual expiry dates and negative activity values.
5. **History windows and evaluation:** which candidate windows and history summaries should become predictors, and which mature outcomes support training, validation and testing.

Potential predictors include recent listening days, listening duration, payment history and latest pre-cutoff auto-renew status. They remain proposals until the prediction cutoff and availability rules are fixed. Keep labels separate from predictor columns.

In [ ]:
findings = []
for row in comparison.to_dict(orient="records"):
    findings.append({
        "source": row["source"], "scope": "Full file",
        "finding": f"{row['rows']:,} labels; {row['churn_percentage']:.2f}% marked churn.",
    })
for row in transaction_results:
    findings.append({
        "source": row["source"], "scope": row["scope"],
        "finding": (f"{row['rows']:,} transaction rows; {row['nonpositive_plan_days']:,} "
                    f"have non-positive plan lengths. Expiry dates range from "
                    f"{row['first_expiry_date']} to {row['last_expiry_date']}."),
    })
for row in [updated_log_result, original_log_result]:
    findings.append({
        "source": row["source"], "scope": row["scope"],
        "finding": (f"{row['rows']:,} scanned rows; {row['negative_duration_rows']:,} "
                    f"have negative listening durations."),
    })
for outcome, row in outcome_comparison.iterrows():
    findings.append({
        "source": "Pre-cutoff sample comparison", "scope": f"{outcome}: {int(row['customers']):,} sampled customers",
        "finding": (f"Median recorded active days: {row['median_recorded_active_days_28d']:.1f}; "
                    f"source-record coverage: {row['source_record_coverage_percentage']:.1f}%."),
    })
findings.append({
    "source": "All-history coverage", "scope": f"{len(cohort_ids):,} sampled customers",
    "finding": (f"{cohort_listening_quality['customers_older_records_only']:,} customers have older "
                f"records but none in the last 28 days; "
                f"{cohort_listening_quality['customers_without_any_source_records']:,} have no earlier records."),
})
for outcome, row in outcome_comparison.iterrows():
    findings.append({
        "source": "Personal-baseline comparison", "scope": f"{outcome}: {int(row['customers_with_defined_ratio']):,} defined ratios",
        "finding": f"Median recent-to-baseline recorded-minutes ratio: {row['median_recent_to_baseline_ratio']:.2f}.",
    })
display(pd.DataFrame(findings))
print("Interpretation: compare groups within each window; read coverage and eligibility counts alongside the medians.")
print("These exploratory summaries do not determine the best predictors or demonstrate causes of churn.")


## 21. Save aggregate outputs

Save the label, profile, transaction and listening summaries to `summary.json`. Figures and daily aggregate tables are saved alongside it. Customer-level previews and source rows are not exported by this cell. The separately generated listening-sample cache contains customer records, stays local and must remain outside Git.

Generated outputs remain outside Git. Clear notebook outputs before committing. No feature pipeline or model is built in this notebook.

In [ ]:
summary = {
    "runtime": "Kaggle" if ON_KAGGLE else "local",
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "label_comparison": comparison.to_dict(orient="records"),
    "profile_rows": profile_rows,
    "profile_quality": profile_quality.to_dict(orient="records"),
    "transaction_summary": transaction_summary.to_dict(orient="records"),
    "listening_summary": listening_summary.to_dict(orient="records"),
    "findings": findings,
    "cohort_eda": {
        "prediction_date": PREDICTION_DATE.date().isoformat(),
        "retained_history": "All available original-source records before cutoff",
        "lookback_days": list(LOOKBACK_DAYS),
        "sample_size": len(cohort_ids), "seed": COHORT_SEED,
        "original_source_rows_scanned": cohort_scan_manifest["scanned_rows"],
        "listening_quality": cohort_listening_quality,
        "window_comparison": window_comparison.to_dict(orient="records"),
        "coverage_comparison": coverage_comparison.reset_index().to_dict(orient="records"),
        "outcome_comparison": outcome_comparison.reset_index().to_dict(orient="records"),
        "renewal_comparison": renewal_comparison.reset_index().to_dict(orient="records"),
    },
    "scope": ("Full labels, profile quality, both transaction files and updated listening logs; "
              f"original listening quality prefix: {ORIGINAL_LOG_PREFIX_ROWS:,} rows; "
              "full original listening scan retains all pre-cutoff history for a fixed customer sample."),
    "models_trained": False,
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
transaction_summary.to_csv(OUTPUT / "transaction_summary.csv", index=False)
listening_summary.to_csv(OUTPUT / "listening_summary.csv", index=False)
listening_daily.to_csv(OUTPUT / "updated_listening_daily.csv", index_label="date")
outcome_comparison.to_csv(OUTPUT / "cohort_outcome_comparison.csv")
renewal_comparison.to_csv(OUTPUT / "cohort_renewal_comparison.csv")
window_comparison.to_csv(OUTPUT / "cohort_window_comparison.csv", index=False)
coverage_comparison.to_csv(OUTPUT / "cohort_coverage_comparison.csv")
print("Saved aggregate tables and figures:", OUTPUT)
print("EDA complete. No feature pipeline or model was built.")
